# 008 Streaming

这是 LangGraph 学习线的第八份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langgraph/streaming

学习目标：

1. 区分上一课的 event streaming 和本课的 stream-mode API
2. 理解 `version="v2"` 的统一 `StreamPart` 格式
3. 学会同时消费 `updates`、`values`、`custom` 等 stream modes
4. 用 fake chat model 跑通 `messages` token streaming
5. 理解 `tasks`、`checkpoints`、`debug` 适合什么场景
6. 判断哪些内部事件适合转换成前端 SSE 业务事件

## 1. Streaming 和 Event Streaming 的关系

上一课学习的是官方新版 event streaming：它把底层事件整理成 typed projections，例如 `stream.messages`、`stream.values`、`stream.output`。

本课学习的是更底层的 stream-mode API：

```text
graph.stream(..., stream_mode="updates")
graph.stream(..., stream_mode=["updates", "custom"], version="v2")
```

直观理解：

| API | 更像什么 | 适合场景 |
| --- | --- | --- |
| event streaming | 已经整理好的业务投影 | 新应用、UI 直接消费多个投影 |
| streaming | 原始运行时事件出口 | 调试、底层观察、自定义 SSE 映射 |

官方文档也建议：新应用优先看 event streaming；当你需要直接访问 graph runtime event 时，再使用本课的 streaming API。

In [96]:
import importlib.metadata
import json

from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langgraph.checkpoint.memory import MemorySaver
from langgraph.config import get_stream_writer
from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict

print("langgraph", importlib.metadata.version("langgraph"))

langgraph 1.2.1


## 2. 定义一个带 custom event 的 graph

这个 graph 有两个节点：

```text
START -> refine_topic -> write_answer -> END
```

每个节点除了返回 state update，还会通过 `get_stream_writer()` 发出自定义进度事件。

In [97]:
class StreamStudyState(TypedDict):
    topic: str
    refined_topic: str
    answer: str


def refine_topic(state: StreamStudyState) -> dict:
    writer = get_stream_writer()
    writer({"stage": "refine_topic", "progress": 40})
    return {"refined_topic": state["topic"] + " + runtime events"}


def write_answer(state: StreamStudyState) -> dict:
    writer = get_stream_writer()
    writer({"stage": "write_answer", "progress": 90})
    return {"answer": "回答主题：" + state["refined_topic"]}


study_builder = StateGraph(StreamStudyState)
study_builder.add_node("refine_topic", refine_topic)
study_builder.add_node("write_answer", write_answer)
study_builder.add_edge(START, "refine_topic")
study_builder.add_edge("refine_topic", "write_answer")
study_builder.add_edge("write_answer", END)

study_graph = study_builder.compile()

initial_state = {
    "topic": "LangGraph streaming",
    "refined_topic": "",
    "answer": "",
}

study_graph.invoke(initial_state)

{'topic': 'LangGraph streaming',
 'refined_topic': 'LangGraph streaming + runtime events',
 'answer': '回答主题：LangGraph streaming + runtime events'}

## 3. `version="v2"` 的统一输出格式

官方文档推荐在 streaming 中使用 `version="v2"`。

原因是 v2 不管你开几个 stream mode，输出都统一成 `StreamPart`：

```python
{
    "type": "updates" | "values" | "messages" | "custom" | "checkpoints" | "tasks" | "debug",
    "ns": (),
    "data": ...,
}
```

这比旧格式更适合写稳定分支逻辑。

In [98]:
for part in study_graph.stream(initial_state, stream_mode="updates", version="v2"):
    print("type=", part["type"], "ns=", part["ns"], "data=", part["data"])

type= updates ns= () data= {'refine_topic': {'refined_topic': 'LangGraph streaming + runtime events'}}
type= updates ns= () data= {'write_answer': {'answer': '回答主题：LangGraph streaming + runtime events'}}


## 4. `updates` 和 `values` 的区别

`updates` 只看节点返回的增量。

`values` 看每一步之后完整 state。

In [99]:
print("updates:")
for part in study_graph.stream(initial_state, stream_mode="updates", version="v2"):
    print(part["data"])

print("\nvalues:")
for part in study_graph.stream(initial_state, stream_mode="values", version="v2"):
    print(part["data"])

updates:
{'refine_topic': {'refined_topic': 'LangGraph streaming + runtime events'}}
{'write_answer': {'answer': '回答主题：LangGraph streaming + runtime events'}}

values:
{'topic': 'LangGraph streaming', 'refined_topic': '', 'answer': ''}
{'topic': 'LangGraph streaming', 'refined_topic': 'LangGraph streaming + runtime events', 'answer': ''}
{'topic': 'LangGraph streaming', 'refined_topic': 'LangGraph streaming + runtime events', 'answer': '回答主题：LangGraph streaming + runtime events'}


## 5. 同时打开多个 stream modes

多个 mode 同时打开时，v2 的优势会更明显。

你只需要判断 `part["type"]`，不用猜返回值到底是 dict、tuple 还是 triple。

In [100]:
for part in study_graph.stream(
    initial_state,
    stream_mode=["updates", "custom"],
    version="v2",
):
    if part["type"] == "custom":
        print("progress:", part["data"])
    elif part["type"] == "updates":
        print("node update:", part["data"])

progress: {'stage': 'refine_topic', 'progress': 40}
node update: {'refine_topic': {'refined_topic': 'LangGraph streaming + runtime events'}}
progress: {'stage': 'write_answer', 'progress': 90}
node update: {'write_answer': {'answer': '回答主题：LangGraph streaming + runtime events'}}


## 6. `messages`：模型 token/message streaming

`messages` 模式用于观察 LangChain chat model 的输出 chunk。

为了不依赖真实 API key，这里使用 `FakeListChatModel`。

重点不是 fake model，而是看清输出结构：

```text
part["data"] = (message_chunk, metadata)
metadata["langgraph_node"] 可以告诉你 token 来自哪个节点
```

In [101]:
class MessageState(TypedDict):
    topic: str
    answer: str


fake_model = FakeListChatModel(responses=["fake streamed answer"])


def call_fake_model(state: MessageState) -> dict:
    response = fake_model.invoke([
        {"role": "user", "content": "回答主题：" + state["topic"]}
    ])
    return {"answer": response.content}


message_graph = (
    StateGraph(MessageState)
    .add_node("call_fake_model", call_fake_model)
    .add_edge(START, "call_fake_model")
    .add_edge("call_fake_model", END)
    .compile()
)

tokens = []
last_metadata = {}
for part in message_graph.stream(
    {"topic": "stream messages", "answer": ""},
    stream_mode="messages",
    version="v2",
):
    if part["type"] == "messages":
        message_chunk, metadata = part["data"]
        last_metadata = metadata
        if message_chunk.content:
            tokens.append(message_chunk.content)
            print(message_chunk.content, end="|")

print("\njoined:", "".join(tokens))
print("node:", last_metadata.get("langgraph_node"))

f|a|k|e| |s|t|r|e|a|m|e|d| |a|n|s|w|e|r|
joined: fake streamed answer
node: call_fake_model


## 7. `tasks`：观察节点任务开始和结束

`tasks` 适合回答：

```text
哪个节点开始执行？
哪个节点执行结束？
结果是什么？有没有 error？
```

官方文档说明 `tasks` 需要 checkpointer。这里使用内存 checkpointer。

In [102]:
checkpoint_graph = study_builder.compile(checkpointer=MemorySaver())
task_config = {"configurable": {"thread_id": "streaming-tasks-demo"}}

for part in checkpoint_graph.stream(
    initial_state,
    config=task_config,
    stream_mode="tasks",
    version="v2",
):
    data = part["data"]
    print(
        "task=", data.get("name"),
        "has_result=", "result" in data,
        "error=", data.get("error"),
    )

task= refine_topic has_result= False error= None
task= refine_topic has_result= True error= None
task= write_answer has_result= False error= None
task= write_answer has_result= True error= None


## 8. `checkpoints`：观察状态快照

`checkpoints` 适合回答：

```text
每一步保存了什么状态？
下一步准备执行哪个节点？
thread_id 下产生了哪些 checkpoint？
```

这和前面 persistence 课程里的 `get_state()` 是一组概念。

In [103]:
checkpoint_config = {"configurable": {"thread_id": "streaming-checkpoints-demo"}}

for part in checkpoint_graph.stream(
    initial_state,
    config=checkpoint_config,
    stream_mode="checkpoints",
    version="v2",
):
    data = part["data"]
    print(
        "step=", data["metadata"].get("step"),
        "next=", data.get("next"),
        "values_keys=", sorted(data.get("values", {}).keys()),
    )

step= -1 next= ['__start__'] values_keys= []
step= 0 next= ['refine_topic'] values_keys= ['answer', 'refined_topic', 'topic']
step= 1 next= ['write_answer'] values_keys= ['answer', 'refined_topic', 'topic']
step= 2 next= [] values_keys= ['answer', 'refined_topic', 'topic']


## 9. `debug`：尽量多地输出运行时信息

`debug` 适合排查问题，但不适合直接暴露给前端用户。

它会包含 task、checkpoint 等信息。产品里通常只把其中一部分转换成业务事件。

In [104]:
for index, part in enumerate(study_graph.stream(initial_state, stream_mode="debug", version="v2")):
    print("type=", part["type"], "data_type=", type(part["data"]).__name__)
    print("data=", part["data"])
    if index >= 3:
        break

type= debug data_type= dict
data= {'step': 1, 'timestamp': '2026-06-08T09:35:27.208125+00:00', 'type': 'task', 'payload': {'id': '7e10de05-1300-3679-f353-0cabe70ba26a', 'name': 'refine_topic', 'input': {'topic': 'LangGraph streaming', 'refined_topic': '', 'answer': ''}, 'triggers': ('branch:to:refine_topic',)}}
type= debug data_type= dict
data= {'step': 1, 'timestamp': '2026-06-08T09:35:27.208544+00:00', 'type': 'task_result', 'payload': {'id': '7e10de05-1300-3679-f353-0cabe70ba26a', 'name': 'refine_topic', 'error': None, 'result': {'refined_topic': 'LangGraph streaming + runtime events'}, 'interrupts': []}}
type= debug data_type= dict
data= {'step': 2, 'timestamp': '2026-06-08T09:35:27.208780+00:00', 'type': 'task', 'payload': {'id': '833e2de5-f981-72b3-c57a-3ba0cc39f9b4', 'name': 'write_answer', 'input': {'topic': 'LangGraph streaming', 'refined_topic': 'LangGraph streaming + runtime events', 'answer': ''}, 'triggers': ('branch:to:write_answer',)}}
type= debug data_type= dict
data= {

## 10. 映射成 SSE 业务事件

底层 stream event 不一定要原样给前端。

更常见的做法是：

| LangGraph stream part | 前端业务事件 |
| --- | --- |
| `custom` progress | `progress` |
| `updates` final node output | `node_update` |
| `messages` token chunk | `answer_delta` |
| `tasks` error | `agent_error` |
| `checkpoints` | 通常只留给内部审计，不直接展示 |

下面只是格式演示，不启动 FastAPI 服务。

In [105]:
def to_sse(event_name: str, payload: dict) -> str:
    return "event: " + event_name + "\n" + "data: " + json.dumps(payload, ensure_ascii=False) + "\n"


for part in study_graph.stream(
    initial_state,
    stream_mode=["updates", "custom"],
    version="v2",
):
    if part["type"] == "custom":
        print(to_sse("progress", part["data"]))
    elif part["type"] == "updates":
        print(to_sse("node_update", part["data"]))

event: progress
data: {"stage": "refine_topic", "progress": 40}

event: node_update
data: {"refine_topic": {"refined_topic": "LangGraph streaming + runtime events"}}

event: progress
data: {"stage": "write_answer", "progress": 90}

event: node_update
data: {"write_answer": {"answer": "回答主题：LangGraph streaming + runtime events"}}



## 11. 本讲练习

请判断下面场景应该用哪个 stream mode：

1. 前端只想知道节点产出了哪些新字段。
2. 开发者想看每一步完整 state。
3. 节点内部想主动上报“正在查询数据库”。
4. UI 要逐 token 展示模型回答。
5. 排查某个节点是否真的开始执行。
6. 审计每一步 checkpoint 里的 state。

参考答案：

1. `updates`
2. `values`
3. `custom`
4. `messages`
5. `tasks` 或 `debug`
6. `checkpoints`

## 12. 本讲小结

这一讲的核心：

```text
Streaming 是 LangGraph 底层运行时事件出口。
version="v2" 让所有 mode 都变成统一 StreamPart，方便稳定消费。
```

你现在应该能看懂：

- `stream_mode="updates"`
- `stream_mode="values"`
- `stream_mode="custom"`
- `stream_mode="messages"`
- `stream_mode="tasks"`
- `stream_mode="checkpoints"`
- `stream_mode="debug"`
- 为什么产品里要把底层事件转换成业务事件

下一讲可以继续学习 Interrupts 或 Time Travel。